# Semana 11 — Embeddings e Semantic Search (Exact vs. HNSW)
**Fase 3: Advanced RAG, Search e Reranking** · Notebook de exercícios

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NetoRibeiro/ai-architect-foundations/blob/main/03-Phase-3-Advanced-RAG/notebooks/semana_11_exercicio.ipynb)

Leitura de conceito desta semana: [Semana-11-Conceitos.md](https://github.com/NetoRibeiro/ai-architect-foundations/blob/main/03-Phase-3-Advanced-RAG/docs/Semana-11-Conceitos.md)

Complete cada trecho marcado com `# [SEU CÓDIGO AQUI]`. As células seguintes dependem das anteriores: execute em ordem. O gabarito está em `semana_11_gabarito.ipynb`.

> **GPU:** em *Ambiente de execução → Alterar o tipo de ambiente de execução*, selecione **T4 GPU** antes de começar.

In [ ]:
!pip -q install sentence-transformers faiss-cpu

In [ ]:
import numpy as np, time, re, matplotlib.pyplot as plt, faiss
from sentence_transformers import SentenceTransformer
MODELO_EMB = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

## Passo 1 — A base de conhecimento
Uma base fictícia de políticas internas, com queries e gabarito de relevância. Ela acompanha toda a Fase 3.

In [ ]:
# --- Corpus de demonstração: base de conhecimento fictícia de uma empresa ---
import random
def corpus_demo(seed=7):
    """Devolve (docs, consultas).
    docs: lista de {id, texto, area, ano}.
    consultas: lista de {q, resposta, relevantes: {doc_id: grau}}; grau 2 = contém a resposta, 1 = mesmo assunto."""
    rnd = random.Random(seed)
    fatos = {
        "ferias": [("O colaborador tem direito a 30 dias de férias após 12 meses de trabalho.", "Quantos dias de descanso anual eu tenho?"),
                   ("As férias podem ser divididas em até três períodos, um deles com no mínimo 14 dias.", "Posso fracionar meu período de descanso?"),
                   ("O pedido de férias deve ser feito no portal RH com 45 dias de antecedência.", "Com que antecedência solicito minha folga anual?")],
        "reembolso": [("Despesas de viagem são reembolsadas em até 10 dias úteis após o envio dos recibos.", "Em quanto tempo recebo de volta o que gastei viajando?"),
                      ("O limite diário de alimentação em viagens nacionais é de R$ 120.", "Qual o teto de gasto com refeição em deslocamento a trabalho?"),
                      ("Corridas de aplicativo só são reembolsadas entre 22h e 6h ou em deslocamentos a clientes.", "A empresa paga meu Uber?")],
        "seguranca": [("Senhas corporativas expiram a cada 90 dias e exigem autenticação em dois fatores.", "De quanto em quanto tempo preciso trocar minha credencial?"),
                      ("Dados de clientes classificados como confidenciais não podem ser enviados a ferramentas externas de IA.", "Posso colar informações de cliente em um chatbot público?"),
                      ("Incidentes de segurança devem ser reportados ao SOC em até 2 horas.", "Qual o prazo para avisar sobre um vazamento?")],
        "ti": [("Notebooks são substituídos a cada 36 meses mediante chamado no service desk.", "Quando posso trocar meu computador?"),
               ("O acesso à VPN exige o cliente GlobalConnect e certificado pessoal.", "Como me conecto à rede da empresa de casa?"),
               ("Licenças de software adicionais precisam de aprovação do gestor e do time de compras.", "Como consigo um programa novo?")],
        "beneficios": [("O plano de saúde cobre dependentes legais sem custo adicional até 21 anos.", "Meu filho pode entrar no convênio médico?"),
                       ("O vale-refeição é creditado no último dia útil de cada mês.", "Em que dia cai o cartão de alimentação?"),
                       ("A empresa subsidia 50% de cursos de idiomas, limitado a R$ 400 por mês.", "Existe ajuda para estudar inglês?")],
    }
    codigos = [("O código de erro VPN-4031 indica certificado pessoal expirado; renove pelo portal de identidades.", "ti", "O que significa o erro VPN-4031?"),
               ("O formulário RH-207 é usado para solicitar licença não remunerada.", "ferias", "Para que serve o formulário RH-207?"),
               ("O centro de custo CC-8812 deve ser usado para despesas de treinamento externo.", "reembolso", "Qual centro de custo uso: CC-8812 é para quê?"),
               ("A política SEG-014 define a classificação de dados em público, interno e confidencial.", "seguranca", "O que diz a política SEG-014?")]
    projetos = ["Atlas", "Boreal", "Cometa", "Delta", "Éden"]
    armadilhas = ["O time do projeto {p} volta de férias em janeiro e retoma as entregas do trimestre.",
                  "No projeto {p}, o reembolso ao cliente foi aprovado após a revisão do contrato.",
                  "A equipe do projeto {p} trocou a senha do ambiente de testes durante a migração.",
                  "O projeto {p} comprou notebooks novos para o laboratório de dados.",
                  "O plano de saúde financeira do projeto {p} prevê corte de 10% nos custos de nuvem."]
    ruido = ["Este procedimento foi revisado pelo comitê interno.", "Dúvidas devem ser encaminhadas ao gestor imediato.",
             "A política vale para todas as unidades do grupo.", "Exceções exigem aprovação formal da diretoria."]
    docs, consultas = [], []
    def novo(texto, area):
        docs.append({"id": len(docs), "texto": texto, "area": area, "ano": rnd.choice([2023, 2024, 2025])}); return docs[-1]["id"]
    por_area = {}
    for area, itens in fatos.items():
        for fato, pergunta in itens:
            i = novo(f"{fato} {rnd.choice(ruido)}", area); por_area.setdefault(area, []).append(i)
            consultas.append({"q": pergunta, "resposta": fato, "relevantes": {i: 2}, "area": area})
    for texto, area, pergunta in codigos:
        i = novo(texto, area); por_area[area].append(i)
        consultas.append({"q": pergunta, "resposta": texto, "relevantes": {i: 2}, "area": area})
    for c in consultas:
        for j in por_area[c["area"]]:
            c["relevantes"].setdefault(j, 1)
    for p in projetos:
        for a in armadilhas:
            novo(a.format(p=p), "projetos")
    return docs, consultas

def manual_demo(docs):
    """Junta os documentos de política em um manual longo em Markdown (para exercícios de chunking)."""
    titulos = {"ferias": "Férias e licenças", "reembolso": "Viagens e reembolsos", "seguranca": "Segurança da informação",
               "ti": "Equipamentos e acesso", "beneficios": "Benefícios"}
    partes = ["# Manual do Colaborador\n\nEste manual reúne as políticas internas vigentes."]
    for area, titulo in titulos.items():
        frases = [d["texto"] for d in docs if d["area"] == area]
        partes.append(f"## {titulo}\n\n" + "\n\n".join(" ".join(frases[i:i + 2]) for i in range(0, len(frases), 2)))
    return "\n\n".join(partes)

docs, consultas = corpus_demo()
print(len(docs), "documentos |", len(consultas), "consultas")
print(docs[0], "\n", consultas[0])

## Passo 2 — Codificar a base
Carregue o modelo e gere a matriz de embeddings **normalizados** dos textos.

In [ ]:
modelo = SentenceTransformer(MODELO_EMB)
# [SEU CÓDIGO AQUI] E = modelo.encode(..., normalize_embeddings=True, convert_to_numpy=True)
print("shape:", E.shape, "| norma da 1ª linha:", round(float(np.linalg.norm(E[0])), 3))

## Passo 3 — Exact search em NumPy
É o seu mini vector database da Semana 2, agora com embeddings de verdade.

In [ ]:
def buscar(q, k=3):
    # [SEU CÓDIGO AQUI] codifique a query, calcule scores e pegue o top-k em idx
    return [(int(i), float(scores[i])) for i in idx]

for q in ["Quantos dias de descanso anual eu tenho?", "A empresa paga meu Uber?"]:
    print("\n>>", q)
    for i, s in buscar(q): print(f"   {s:.3f}  {docs[i]['texto'][:90]}")

## Passo 4 — Semântica vs. palavras em comum
As queries são paráfrases: quase não repetem as palavras do documento. Compare com um buscador lexical ingênuo (word overlap).

In [ ]:
def palavras(t): return set(re.findall(r"\w+", t.lower()))
def buscar_lexical(q, k=3):
    pq = palavras(q)
    scores = np.array([len(pq & palavras(d["texto"])) / (len(pq | palavras(d["texto"])) or 1) for d in docs])
    return [(int(i), float(scores[i])) for i in np.argsort(-scores)[:k]]

def taxa_de_acerto(fn, k):
    # [SEU CÓDIGO AQUI] fração das queries em que o documento de grau 2 aparece no top-k

for k in (1, 3):
    print(f"hit@{k} | semântico: {taxa_de_acerto(buscar, k):.2f} | lexical: {taxa_de_acerto(buscar_lexical, k):.2f}")

Em quais queries o lexical ganha? Olhe as que têm códigos como `VPN-4031`. Guarde essa observação para a Semana 14.

## Passo 5 — Cross-lingual search

In [ ]:
for q in ["How many vacation days do I get?", "¿Cada cuánto tengo que cambiar mi contraseña?"]:
    i, s = buscar(q, 1)[0]; print(f"{q}\n   -> {s:.3f} {docs[i]['texto'][:80]}")

## Passo 6 — Quando a base cresce: exato vs. HNSW
Para medir indexes, precisamos de volume. Geramos 50 mil vetores sintéticos em aglomerados (parecido com embeddings reais).

In [ ]:
rng = np.random.default_rng(0); N, d, nq = 50_000, 384, 200
centros = rng.standard_normal((200, d)).astype("float32")
def amostra(n):
    X = centros[rng.integers(0, 200, n)] + 0.6 * rng.standard_normal((n, d)).astype("float32")
    return X / np.linalg.norm(X, axis=1, keepdims=True)
X, Q = amostra(N), amostra(nq)

# [SEU CÓDIGO AQUI] crie o exact index (IndexFlatIP) e adicione X
t0 = time.perf_counter(); _, gabarito = flat.search(Q, 10); t_flat = (time.perf_counter() - t0) / nq * 1000
print(f"exato: {t_flat:.3f} ms por consulta")

Construa um `IndexHNSWFlat` com `M=32`, métrica de produto interno e `efConstruction=80`.

In [ ]:
t0 = time.perf_counter()
# [SEU CÓDIGO AQUI] crie hnsw, defina efConstruction e adicione X
print(f"construção do HNSW: {time.perf_counter() - t0:.1f}s")

def recall_at_k(encontrados, gabarito):
    # [SEU CÓDIGO AQUI] média da fração do gabarito recuperada em cada query

## Passo 7 — A curva recall × latência

In [ ]:
res = []
for ef in (8, 16, 32, 64, 128, 256):
    hnsw.hnsw.efSearch = ef
    t0 = time.perf_counter(); _, I = hnsw.search(Q, 10); ms = (time.perf_counter() - t0) / nq * 1000
    res.append((ef, recall_at_k(I, gabarito), ms)); print(f"efSearch={ef:4d} | recall@10={res[-1][1]:.3f} | {ms:.3f} ms")
plt.plot([r[2] for r in res], [r[1] for r in res], "o-"); plt.axvline(t_flat, ls="--", c="gray", label="busca exata")
for ef, r, ms in res: plt.annotate(f"ef={ef}", (ms, r))
plt.xlabel("ms por consulta"); plt.ylabel("recall@10"); plt.legend(); plt.show()

Qual `efSearch` você escolheria para um requisito de recall ≥ 0,95? Com 50 mil vetores, o HNSW compensa em relação à exact search? E com 50 milhões?

## Passo 8 — Dimensionando memória

In [ ]:
def memoria_gb(n, dim, M=32):
    # [SEU CÓDIGO AQUI] devolva (GB de vetores float32, GB aproximados do graph HNSW)

for n, dim in [(50_000, 384), (10_000_000, 768), (100_000_000, 1024)]:
    v, g = memoria_gb(n, dim); print(f"N={n:>11,} d={dim:4d} | vetores {v:7.2f} GB | grafo ≈ {g:6.2f} GB")